# SteelVision v2 — classes raras, resolução nativa e retomada
**Flávia Gaia · laboratório 1 de 4 · PyTorch**

Esta versão implementa amostragem balanceada, BCE ponderada com pesos limitados, recortes nativos,
encoder ResNet18 pré-treinado opcional, checkpoint de retomada e calibração por classe somente na validação.
Validação, calibração, teste e inferência isolada usam o mesmo mosaico na resolução original.

**Protocolo:** preservar os splits históricos por hash. O teste foi examinado no baseline e será denominado
**teste histórico conhecido**, não um novo conjunto cego. Não ajustar configurações olhando seu resultado.
Uma comprovação externa independente exige novas imagens de outro lote/período, que não temos aqui.

O candidato combina mudanças; eventual ganho não pode ser atribuído isoladamente a uma delas.
As ablações devem usar o mesmo split e orçamento, escolhidas na validação antes de consultar o teste.

**Kaggle:** dados Severstal anexados, GPU e Internet ativadas para baixar os pesos oficiais ImageNet.
Não há fallback silencioso para pesos aleatórios. `PRETRAINED=False` permite execução sem download.
Não submeter automaticamente à competição.

[Baseline e resultados anteriores](https://github.com/flaviagaia/industrial-vision-mlops) ·
[Severstal](https://www.kaggle.com/competitions/severstal-steel-defect-detection/data)


## 1. Configuração antes da execução
O orçamento de quatro épocas é um experimento inicial. O teste não participa da seleção do checkpoint
nem da busca de limiares. Para retomar: mantenha a mesma pasta e configuração e indique `RESUME_FROM`.
A retomada é exata na fronteira de época; falha no meio de uma época repete essa época inteira.


In [ ]:
from pathlib import Path
USE_SYNTHETIC = False
RUN_NAME = "severstal_v2_native_resnet18_01"
EPOCHS = 4
BATCH_SIZE = 8
ARCHITECTURE = "resnet18_unet"
PRETRAINED = True
BALANCED_SAMPLING = True
WEIGHTED_LOSS = True
RESUME_FROM = None  # ex.: Path("/kaggle/working/.../run/last.pt")
DATA_DIR_OVERRIDE = None
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
OUTPUT = BASE / "steelvision_artifacts" / RUN_NAME
PACKAGE_ROOT = BASE / "steelvision_v2_source"
PACKAGE = PACKAGE_ROOT / "steelvision"
PACKAGE.mkdir(parents=True, exist_ok=True)


## 2. Código incorporado do repositório

### __init__.py

In [ ]:
source = r'''"""SteelVision: baseline de segmentação multirrótulo de defeitos em aço."""

__version__ = "0.1.0"
'''
(PACKAGE / '__init__.py').write_text(source, encoding='utf-8')

### advanced.py

In [ ]:
source = r'''"""Protocolo v2: treino balanceado, retomada, calibração e avaliação nativa."""

import csv
import hashlib
import json
import os
import random
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader

from .advanced_data import (
    EpochSampler,
    NativePatchDataset,
    original_sample,
    records_for,
    training_statistics,
    verify_integrity,
)
from .advanced_inference import ThresholdSearch, native_probabilities
from .data import encode_rle, sha256_file
from .engine import choose_device, seed_everything
from .metrics import SegmentationMetrics
from .model import segmentation_loss
from .transfer import build_model
from .visualize import plot_history, save_comparison


@dataclass
class AdvancedConfig:
    epochs: int = 6
    batch_size: int = 8
    patch_height: int = 256
    patch_width: int = 400
    architecture: str = "resnet18_unet"
    pretrained: bool = True
    base_channels: int = 16
    learning_rate: float = 0.0003
    seed: int = 42
    num_workers: int = 2
    patience: int = 4
    mixed_precision: bool = True
    balanced_sampling: bool = True
    weighted_loss: bool = True
    max_pos_weight: float = 20.0
    positive_crop_probability: float = 0.5
    device: str = "auto"
    data_kind: str = "severstal"


def atomic_save(value, path):
    path = Path(path)
    temp = path.with_suffix(path.suffix + ".tmp")
    torch.save(value, temp)
    os.replace(temp, path)


def source_hash():
    digest = hashlib.sha256()
    for path in sorted(Path(__file__).parent.glob("*.py")):
        digest.update(path.name.encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


def rng_state():
    state = np.random.get_state()
    return {
        "python": random.getstate(),
        "torch": torch.get_rng_state(),
        "numpy": [state[0], state[1].tolist(), state[2], state[3], state[4]],
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
    }


def restore_rng(state):
    random.setstate(state["python"])
    torch.set_rng_state(state["torch"])
    value = state["numpy"]
    np.random.set_state((value[0], np.array(value[1], dtype=np.uint32), *value[2:]))
    if state["cuda"] and torch.cuda.is_available():
        torch.cuda.set_rng_state_all(state["cuda"])


def load_model(checkpoint, device="auto"):
    saved = torch.load(checkpoint, map_location="cpu", weights_only=True)
    config = AdvancedConfig(**saved["config"])
    model = build_model(config, initialize_pretrained=False)
    model.load_state_dict(saved["model_state"])
    target = choose_device(device)
    return model.to(target).eval(), config, saved, target


def native_pass(model, config, data_dir, manifest, split, device, thresholds=0.5, search=None):
    meter = SegmentationMetrics()
    records = records_for(manifest, split)
    if not records:
        raise ValueError(f"Split vazio: {split}")
    started = time.perf_counter()
    for index, record in enumerate(records, 1):
        image, target = original_sample(data_dir, record)
        prob = native_probabilities(
            model,
            image,
            (config.patch_height, config.patch_width),
            device,
            config.batch_size,
            config.mixed_precision,
        )
        meter.update(prob[None], target[None], [record["image_id"]], thresholds)
        if search is not None:
            search.update(prob, target)
        if index % 400 == 0 or index == len(records):
            print(
                json.dumps(
                    {
                        "phase": f"{split}_native",
                        "images": index,
                        "total": len(records),
                        "seconds": round(time.perf_counter() - started, 1),
                    }
                ),
                flush=True,
            )
    return meter


def train_advanced(data_dir, manifest, output_dir, config, resume=None):
    output = Path(output_dir)
    if config.epochs < 1 or min(config.patch_height, config.patch_width) < 32:
        raise ValueError("Épocas positivas e patches de pelo menos 32 pixels são obrigatórios")
    if (
        config.batch_size < 1
        or config.num_workers < 0
        or config.learning_rate <= 0
        or config.max_pos_weight < 1
    ):
        raise ValueError("Batch, workers, learning rate ou peso máximo inválido")
    if not 0 <= config.positive_crop_probability <= 1:
        raise ValueError("Probabilidade de recorte inválida")
    integrity = verify_integrity(data_dir, manifest, {"train", "val"})
    if (output / "last.pt").exists() and resume is None:
        raise FileExistsError("Execução existe; forneça resume=last.pt ou outra pasta")
    if resume is not None and not (output / "best.pt").exists():
        raise ValueError("Retome na pasta original, preservando best.pt e last.pt")
    output.mkdir(parents=True, exist_ok=True)
    device = choose_device(config.device)
    seed_everything(config.seed)
    statistics = training_statistics(manifest, config.max_pos_weight)
    dataset = NativePatchDataset(
        data_dir,
        manifest,
        (config.patch_height, config.patch_width),
        config.seed,
        config.positive_crop_probability,
    )
    saved = torch.load(resume, map_location="cpu", weights_only=True) if resume else None
    if saved:
        if config.epochs < saved["epoch"]:
            raise ValueError("O orçamento não pode ser menor que a época já concluída")
        previous, current = saved["config"].copy(), asdict(config)
        previous.pop("epochs")
        current.pop("epochs")
        if previous != current or saved["manifest_sha256"] != integrity["manifest_sha256"]:
            raise ValueError("Configuração ou manifesto incompatível com a retomada")
        if saved["source_sha256"] != source_hash():
            raise ValueError("Código alterado: não é uma retomada exata")
    model = build_model(config, initialize_pretrained=not bool(saved)).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    amp = config.mixed_precision and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    pos_weight = (
        torch.tensor(statistics["pos_weight"], device=device).reshape(4, 1, 1)
        if config.weighted_loss
        else None
    )
    history, best, stale, initial_epoch, elapsed = [], -1.0, 0, 0, 0.0
    if saved:
        model.load_state_dict(saved["model_state"])
        optimizer.load_state_dict(saved["optimizer_state"])
        scaler.load_state_dict(saved["scaler_state"])
        history, best, stale = saved["history"], saved["best_score"], saved["stale"]
        initial_epoch, elapsed = saved["epoch"], saved["elapsed_seconds"]
        restore_rng(saved["rng_state"])
    metadata = {
        "protocol": "v2_native",
        "config": asdict(config),
        **integrity,
        "source_sha256": source_hash(),
        "torch": str(torch.__version__),
        "device": str(device),
        "amp_enabled": amp,
        "resume_from_epoch": initial_epoch,
        "historical_test_already_inspected": True,
        "threshold_during_selection": 0.5,
    }
    (output / "manifest.json").write_text(Path(manifest).read_text())
    (output / "training_statistics.json").write_text(json.dumps(statistics, indent=2))
    started = time.perf_counter()
    for epoch in range(initial_epoch + 1, config.epochs + 1):
        if stale >= config.patience:
            break
        sampler = EpochSampler(
            statistics["sample_weights"], config.seed, epoch, config.balanced_sampling
        )
        batches = DataLoader(
            dataset,
            batch_size=config.batch_size,
            sampler=sampler,
            num_workers=config.num_workers,
            pin_memory=device.type == "cuda",
            generator=torch.Generator().manual_seed(config.seed + epoch),
        )
        model.train()
        loss_sum = count = 0
        epoch_start = time.perf_counter()
        for step, (image, target, _) in enumerate(batches, 1):
            image, target = (
                image.to(device, non_blocking=True),
                target.to(device, non_blocking=True),
            )
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                loss = segmentation_loss(model(image), target, pos_weight)
            if not torch.isfinite(loss):
                raise RuntimeError("Loss não finita")
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += float(loss.detach()) * len(image)
            count += len(image)
            if step % 100 == 0 or step == len(batches):
                print(
                    json.dumps(
                        {
                            "phase": "train_v2",
                            "epoch": epoch,
                            "step": step,
                            "steps": len(batches),
                            "seconds": round(time.perf_counter() - epoch_start, 1),
                        }
                    ),
                    flush=True,
                )
        validation = native_pass(model, config, data_dir, manifest, "val", device).summary()
        score = validation["macro_dice"] or 0.0
        row = {
            "epoch": epoch,
            "train_loss": loss_sum / count,
            "val_loss": None,
            "val_macro_dice": score,
        }
        history.append(row)
        if score > best:
            best, stale = score, 0
            atomic_save(
                {
                    "model_state": model.state_dict(),
                    "config": asdict(config),
                    "epoch": epoch,
                    "val_metrics": validation,
                    **integrity,
                    "source_sha256": source_hash(),
                    "thresholds": [0.5] * 4,
                },
                output / "best.pt",
            )
        else:
            stale += 1
        atomic_save(
            {
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "scaler_state": scaler.state_dict(),
                "config": asdict(config),
                "epoch": epoch,
                "history": history,
                "best_score": best,
                "stale": stale,
                "rng_state": rng_state(),
                "elapsed_seconds": elapsed + time.perf_counter() - started,
                **integrity,
                "source_sha256": source_hash(),
            },
            output / "last.pt",
        )
        (output / "history.json").write_text(json.dumps(history, indent=2))
        print(json.dumps(row), flush=True)
    metadata.update(
        {
            "epochs_completed": len(history),
            "best_val_macro_dice": best,
            "elapsed_seconds": elapsed + time.perf_counter() - started,
        }
    )
    (output / "run.json").write_text(json.dumps(metadata, indent=2))
    plot_history(history, output / "learning_curves.png")
    return output / "best.pt"


def calibrate(data_dir, manifest, checkpoint, output_dir):
    output = Path(output_dir)
    if (output / "calibration.json").exists():
        raise FileExistsError("Calibração existe; não sobrescrever")
    model, config, saved, device = load_model(checkpoint)
    integrity = verify_integrity(data_dir, manifest, {"val"})
    if integrity["manifest_sha256"] != saved["manifest_sha256"]:
        raise ValueError("Manifesto incompatível")
    search = ThresholdSearch()
    fixed = native_pass(model, config, data_dir, manifest, "val", device, search=search).summary()
    result = {
        **search.result(),
        "fixed_threshold_metrics": fixed,
        "checkpoint_sha256": sha256_file(checkpoint),
        **integrity,
    }
    output.mkdir(parents=True, exist_ok=True)
    (output / "calibration.json").write_text(json.dumps(result, indent=2))
    saved["thresholds"] = result["thresholds"]
    saved["calibration_sha256"] = sha256_file(output / "calibration.json")
    atomic_save(saved, output / "calibrated.pt")
    return output / "calibrated.pt"


def evaluate_advanced(data_dir, manifest, checkpoint, output_dir, split="test"):
    output = Path(output_dir)
    if (output / "metrics.json").exists():
        raise FileExistsError("Avaliação existe; preserve o resultado")
    model, config, saved, device = load_model(checkpoint)
    integrity = verify_integrity(data_dir, manifest, {split})
    if integrity["manifest_sha256"] != saved["manifest_sha256"]:
        raise ValueError("Manifesto incompatível")
    thresholds = saved.get("thresholds", [0.5] * 4)
    meter = native_pass(model, config, data_dir, manifest, split, device, thresholds)
    metrics = {
        **meter.summary(),
        **integrity,
        "data_kind": config.data_kind,
        "split": split,
        "evaluation_role": "historical_known_test" if split == "test" else "development",
        "resolution": "original",
        "postprocessing": "native_tiles_mean_probabilities_then_threshold",
        "thresholds": thresholds,
        "best_epoch": saved["epoch"],
        "checkpoint_sha256": sha256_file(checkpoint),
        "calibration_sha256": saved.get("calibration_sha256"),
    }
    output.mkdir(parents=True, exist_ok=True)
    (output / "metrics.json").write_text(json.dumps(metrics, indent=2, allow_nan=False))
    with (output / "per_image_class.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(meter.rows[0]))
        writer.writeheader()
        writer.writerows(meter.rows)
    failures = sorted(
        [r for r in meter.rows if r["fp_pixels"] + r["fn_pixels"]], key=lambda r: r["dice"]
    )
    selected = list(dict.fromkeys(r["image_id"] for r in failures))[:4]
    lookup = {r["image_id"]: r for r in records_for(manifest, split)}
    for index, name in enumerate(selected):
        image, target = original_sample(data_dir, lookup[name])
        prob = native_probabilities(
            model,
            image,
            (config.patch_height, config.patch_width),
            device,
            config.batch_size,
            config.mixed_precision,
        )
        save_comparison(
            image,
            target,
            prob,
            output / f"prediction_{index:02d}.png",
            f"v2 | {split} | {name}",
            np.array(thresholds).reshape(4, 1, 1),
        )
    return metrics


@torch.inference_mode()
def predict_advanced(image_path, checkpoint, output_dir):
    model, config, saved, device = load_model(checkpoint)
    with Image.open(image_path) as image:
        array = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    probabilities = native_probabilities(
        model,
        array,
        (config.patch_height, config.patch_width),
        device,
        config.batch_size,
        config.mixed_precision,
    )
    thresholds = np.array(saved.get("thresholds", [0.5] * 4)).reshape(4, 1, 1)
    masks = probabilities >= thresholds
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(output / "prediction.npz", masks=masks)
    payload = {
        "image_id": Path(image_path).name,
        "resolution": "original",
        "thresholds": thresholds.ravel().tolist(),
        "checkpoint_sha256": sha256_file(checkpoint),
        "classes": [{"class_id": k + 1, "encoded_pixels": encode_rle(masks[k])} for k in range(4)],
    }
    (output / "prediction.json").write_text(json.dumps(payload, indent=2))
    save_comparison(
        array, None, probabilities, output / "prediction.png", "v2 inferência nativa", thresholds
    )
    return payload
'''
(PACKAGE / 'advanced.py').write_text(source, encoding='utf-8')

### advanced_data.py

In [ ]:
source = r'''"""Integridade, balanceamento e recortes nativos sem cruzar os splits históricos."""

import json
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageOps
from torch.utils.data import Dataset, Sampler

from .data import decode_rle, sha256_file


def records_for(manifest, split):
    return [r for r in json.loads(Path(manifest).read_text())["records"] if r["split"] == split]


def verify_integrity(data_dir, manifest, splits):
    """Verifica conteúdo, não só dimensões; nunca reescreve um manifesto existente."""
    payload = json.loads(Path(manifest).read_text())
    if sha256_file(Path(data_dir) / "train.csv") != payload["csv_sha256"]:
        raise ValueError("CSV alterado após a criação do manifesto")
    hashes = {}
    for record in payload["records"]:
        previous = hashes.setdefault(record["sha256"], record["split"])
        if previous != record["split"]:
            raise ValueError("Vazamento: hash presente em mais de um split")
        if record["split"] in splits:
            path = Path(data_dir) / "train_images" / record["image_id"]
            if sha256_file(path) != record["sha256"]:
                raise ValueError(f"Imagem alterada: {record['image_id']}")
    return {"manifest_sha256": sha256_file(manifest), "verified_splits": sorted(splits)}


def original_sample(data_dir, record):
    with Image.open(Path(data_dir) / "train_images" / record["image_id"]) as image:
        image = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    masks = np.stack([decode_rle(r, record["height"], record["width"]) for r in record["rles"]])
    return image, masks


def training_statistics(manifest, max_pos_weight=20.0):
    """Todos os pesos são calculados exclusivamente nos registros de treino."""
    records = records_for(manifest, "train")
    if not records:
        raise ValueError("Treino vazio")
    present = np.array([[bool(rle) for rle in r["rles"]] for r in records])
    counts = present.sum(0)
    positive = np.zeros(4, dtype=np.float64)
    total = 0
    for record in records:
        total += record["height"] * record["width"]
        for k, rle in enumerate(record["rles"]):
            if rle:
                positive[k] += decode_rle(rle, record["height"], record["width"]).sum()
    # A raiz e o limite evitam multiplicadores extremos ao combinar amostragem e loss.
    pos_weight = np.clip(np.sqrt((total - positive) / np.maximum(positive, 1)), 1, max_pos_weight)
    inverse = np.sqrt(len(records) / np.maximum(counts, 1))
    sample_weights = np.where(present, inverse[None, :], 0).max(1)
    sample_weights[~present.any(1)] = 1.0
    sample_weights = np.minimum(sample_weights, 10.0)
    return {
        "source_split": "train",
        "positive_images": counts.tolist(),
        "positive_pixels": positive.astype(np.int64).tolist(),
        "pos_weight": pos_weight.tolist(),
        "sample_weights": sample_weights.tolist(),
        "formula": "sqrt(negative/positive) clipped; multilabel max sqrt(N/class_count)",
    }


class EpochSampler(Sampler):
    """Cada desenho inclui epoch e ocorrência: recortes determinísticos e distintos."""

    def __init__(self, weights, seed, epoch, balanced=True):
        self.weights = torch.as_tensor(weights, dtype=torch.double)
        self.seed, self.epoch, self.balanced = seed, epoch, balanced

    def __iter__(self):
        generator = torch.Generator().manual_seed(self.seed + self.epoch * 100003)
        if self.balanced:
            order = torch.multinomial(self.weights, len(self.weights), True, generator=generator)
        else:
            order = torch.randperm(len(self.weights), generator=generator)
        return iter((int(index), draw, self.epoch) for draw, index in enumerate(order))

    def __len__(self):
        return len(self.weights)


class NativePatchDataset(Dataset):
    def __init__(
        self, data_dir, manifest, patch_size=(256, 400), seed=42, positive_crop_probability=0.5
    ):
        self.data_dir = Path(data_dir)
        self.records = records_for(manifest, "train")
        self.patch_size, self.seed = tuple(patch_size), seed
        self.positive_crop_probability = positive_crop_probability

    def __len__(self):
        return len(self.records)

    def __getitem__(self, key):
        index, draw, epoch = key if isinstance(key, tuple) else (key, key, 0)
        record = self.records[index]
        image, masks = original_sample(self.data_dir, record)
        rng = np.random.default_rng(np.random.SeedSequence([self.seed, epoch, draw, index]))
        ph, pw = self.patch_size
        h, w = image.shape[:2]
        if h < ph or w < pw:
            image = np.pad(image, ((0, max(0, ph - h)), (0, max(0, pw - w)), (0, 0)), mode="edge")
            masks = np.pad(masks, ((0, 0), (0, max(0, ph - h)), (0, max(0, pw - w))))
            h, w = image.shape[:2]
        top, left = int(rng.integers(h - ph + 1)), int(rng.integers(w - pw + 1))
        available = np.flatnonzero(masks.reshape(4, -1).any(1))
        if len(available) and rng.random() < self.positive_crop_probability:
            k = int(rng.choice(available))
            points = np.argwhere(masks[k])
            cy, cx = points[int(rng.integers(len(points)))]
            top = int(np.clip(cy - int(rng.integers(ph)), 0, h - ph))
            left = int(np.clip(cx - int(rng.integers(pw)), 0, w - pw))
        image = Image.fromarray((image[top : top + ph, left : left + pw] * 255).astype(np.uint8))
        target = masks[:, top : top + ph, left : left + pw].copy()
        if rng.random() < 0.5:
            image, target = ImageOps.mirror(image), target[:, :, ::-1]
        if rng.random() < 0.5:
            image, target = ImageOps.flip(image), target[:, ::-1, :]
        image = ImageEnhance.Brightness(image).enhance(float(rng.uniform(0.8, 1.2)))
        array = np.asarray(image, dtype=np.float32).transpose(2, 0, 1) / 255.0
        return (
            torch.from_numpy(array.copy()),
            torch.from_numpy(target.copy().astype(np.float32)),
            record["image_id"],
        )
'''
(PACKAGE / 'advanced_data.py').write_text(source, encoding='utf-8')

### advanced_inference.py

In [ ]:
source = r'''"""Mesmo mosaico de inferência nativa para validação, calibração, teste e uso isolado."""

import numpy as np
import torch
from torch.nn import functional as F


def starts(length, patch):
    return sorted(set([*range(0, max(length - patch, 0) + 1, patch), max(0, length - patch)]))


@torch.inference_mode()
def native_probabilities(model, image, patch_size, device, batch_size=4, amp=True):
    h, w = image.shape[:2]
    ph, pw = patch_size
    padded = np.pad(image, ((0, max(0, ph - h)), (0, max(0, pw - w)), (0, 0)), mode="edge")
    positions = [(y, x) for y in starts(padded.shape[0], ph) for x in starts(padded.shape[1], pw)]
    sums = np.zeros((4, *padded.shape[:2]), dtype=np.float32)
    counts = np.zeros(padded.shape[:2], dtype=np.float32)
    model.eval()
    for i in range(0, len(positions), batch_size):
        selected = positions[i : i + batch_size]
        tiles = np.stack([padded[y : y + ph, x : x + pw].transpose(2, 0, 1) for y, x in selected])
        tensor = torch.from_numpy(tiles.copy()).to(device)
        with torch.autocast(
            device_type=device.type, dtype=torch.float16, enabled=amp and device.type == "cuda"
        ):
            logits = model(tensor)
        if logits.shape[-2:] != (ph, pw):
            logits = F.interpolate(
                logits.float(), size=(ph, pw), mode="bilinear", align_corners=False
            )
        probabilities = logits.float().sigmoid().cpu().numpy()
        for (y, x), prob in zip(selected, probabilities):
            sums[:, y : y + ph, x : x + pw] += prob
            counts[y : y + ph, x : x + pw] += 1
    if not counts.all():
        raise RuntimeError("Mosaico com pixels sem cobertura")
    return (sums / counts[None])[:, :h, :w]


class ThresholdSearch:
    """Contagens exatas por limiar sem guardar todas as probabilidades do dataset."""

    def __init__(self, grid=None):
        self.grid = np.asarray(
            grid if grid is not None else np.arange(0.05, 1.0, 0.05), dtype=np.float64
        )
        if self.grid.ndim != 1 or not len(self.grid) or np.any(np.diff(self.grid) <= 0):
            raise ValueError("Grade deve ser crescente e não vazia")
        self.positive = np.zeros((4, len(self.grid) + 1), dtype=np.int64)
        self.negative = self.positive.copy()

    def update(self, probabilities, target):
        for k in range(4):
            buckets = np.searchsorted(self.grid, probabilities[k].ravel(), side="right")
            truth = target[k].ravel() > 0
            self.positive[k] += np.bincount(buckets[truth], minlength=len(self.grid) + 1)
            self.negative[k] += np.bincount(buckets[~truth], minlength=len(self.grid) + 1)

    def result(self):
        output = []
        for k in range(4):
            tp = np.cumsum(self.positive[k][::-1])[::-1][1:]
            fp = np.cumsum(self.negative[k][::-1])[::-1][1:]
            fn = self.positive[k].sum() - tp
            dice = np.divide(
                2 * tp,
                2 * tp + fp + fn,
                out=np.zeros_like(tp, dtype=float),
                where=2 * tp + fp + fn > 0,
            )
            # Se não há positivos de referência, não ajustar um limiar sem suporte.
            best = int(np.flatnonzero(dice == dice.max())[-1]) if self.positive[k].sum() else None
            output.append(
                {
                    "class_id": k + 1,
                    "threshold": float(self.grid[best]) if best is not None else 0.5,
                    "validation_dice": float(dice[best]) if best is not None else None,
                    "positive_pixels": int(self.positive[k].sum()),
                }
            )
        return {
            "source_split": "val",
            "objective": "global Dice per class; ties prefer higher threshold",
            "grid": self.grid.tolist(),
            "classes": output,
            "thresholds": [r["threshold"] for r in output],
        }
'''
(PACKAGE / 'advanced_inference.py').write_text(source, encoding='utf-8')

### cli.py

In [ ]:
source = r'''"""Interface de linha de comando para o projeto de inspeção de aço."""

import argparse
import json
from pathlib import Path


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    prepare = commands.add_parser("prepare", help="Validar RLE e dividir imagens por hash")
    prepare.add_argument("--data", required=True)
    prepare.add_argument("--out", required=True)
    prepare.add_argument("--seed", type=int, default=42)
    eda = commands.add_parser("eda", help="Exploração das classes e máscaras")
    eda.add_argument("--data", required=True)
    eda.add_argument("--manifest", required=True)
    eda.add_argument("--out", required=True)
    train = commands.add_parser("train", help="Treinar U-Net e selecionar na validação")
    train.add_argument("--data", required=True)
    train.add_argument("--manifest", required=True)
    train.add_argument("--out", required=True)
    train.add_argument("--epochs", type=int, default=10)
    train.add_argument("--batch-size", type=int, default=4)
    train.add_argument("--height", type=int, default=128)
    train.add_argument("--width", type=int, default=800)
    train.add_argument("--base-channels", type=int, default=16)
    train.add_argument("--seed", type=int, default=42)
    train.add_argument("--device", default="auto", choices=["auto", "cpu", "cuda", "mps"])
    train.add_argument("--num-workers", type=int, default=0)
    evaluate = commands.add_parser("evaluate", help="Avaliar no teste reservado")
    evaluate.add_argument("--data", required=True)
    evaluate.add_argument("--manifest", required=True)
    evaluate.add_argument("--checkpoint", required=True)
    evaluate.add_argument("--out", required=True)
    evaluate.add_argument("--device", default="auto")
    predict = commands.add_parser("predict", help="Inferência em uma imagem")
    predict.add_argument("--image", required=True)
    predict.add_argument("--checkpoint", required=True)
    predict.add_argument("--out", required=True)
    predict.add_argument("--device", default="auto")
    demo = commands.add_parser("demo", help="Executar fluxo completo com dados SINTÉTICOS")
    demo.add_argument("--out", default="artifacts/synthetic_demo")
    demo.add_argument("--epochs", type=int, default=2)
    args = parser.parse_args(argv)
    if args.command == "prepare":
        from .data import prepare_manifest

        print(prepare_manifest(args.data, args.out, args.seed))
    elif args.command == "eda":
        from .visualize import explore

        print(json.dumps(explore(args.data, args.manifest, args.out), indent=2))
    elif args.command == "train":
        from .engine import Config, train

        config = Config(
            epochs=args.epochs,
            batch_size=args.batch_size,
            height=args.height,
            width=args.width,
            base_channels=args.base_channels,
            seed=args.seed,
            device=args.device,
            num_workers=args.num_workers,
            data_kind="synthetic" if (Path(args.data) / "SYNTHETIC.txt").exists() else "severstal",
        )
        print(train(args.data, args.manifest, args.out, config))
    elif args.command == "evaluate":
        from .engine import evaluate

        print(
            json.dumps(
                evaluate(args.data, args.manifest, args.checkpoint, args.out, device=args.device),
                indent=2,
            )
        )
    elif args.command == "predict":
        from .engine import predict

        result = predict(args.image, args.checkpoint, args.out, args.device)
        print(f"Predição salva: {result['image_id']}")
    elif args.command == "demo":
        import torch

        from .data import prepare_manifest
        from .engine import Config, evaluate, predict, train
        from .synthetic import generate_synthetic
        from .visualize import explore

        # Smoke test pequeno e previsível mesmo sem acelerador.
        torch.set_num_threads(2)
        root = Path(args.out)
        data = generate_synthetic(root / "data")
        manifest = prepare_manifest(data, root / "prepared")
        explore(data, manifest, root / "eda", size=(64, 256))
        config = Config(
            epochs=args.epochs,
            height=64,
            width=256,
            base_channels=8,
            device="cpu",
            data_kind="synthetic",
            batch_size=4,
        )
        checkpoint = train(data, manifest, root / "run", config)
        evaluate(data, manifest, checkpoint, root / "evaluation", device="cpu")
        predict(
            next((data / "train_images").glob("*.png")),
            checkpoint,
            root / "inference",
            device="cpu",
        )
        print(f"DEMO SINTÉTICA concluída: {root.resolve()}")


if __name__ == "__main__":
    main()
'''
(PACKAGE / 'cli.py').write_text(source, encoding='utf-8')

### data.py

In [ ]:
source = r'''"""Leitura Severstal, RLE por coluna e divisão antes de qualquer transformação."""

import csv
import hashlib
import json
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageEnhance, ImageOps
from torch.utils.data import Dataset

CLASSES = (1, 2, 3, 4)


def decode_rle(value, height, width):
    """Decodifica RLE Severstal: índices a partir de 1, ordem Fortran (colunas)."""
    mask = np.zeros(height * width, dtype=np.uint8)
    if value is None or not str(value).strip():
        return mask.reshape((height, width), order="F")
    try:
        runs = np.asarray(str(value).split(), dtype=np.int64)
    except ValueError as exc:
        raise ValueError("RLE deve conter inteiros") from exc
    if len(runs) % 2:
        raise ValueError("RLE deve conter pares início/comprimento")
    starts, lengths = runs[0::2] - 1, runs[1::2]
    if np.any(starts < 0) or np.any(lengths <= 0) or np.any(starts + lengths > mask.size):
        raise ValueError("RLE fora dos limites da imagem")
    for start, length in zip(starts, lengths):
        mask[start : start + length] = 1
    return mask.reshape((height, width), order="F")


def encode_rle(mask):
    """Inverso de decode_rle; usado também na demonstração sintética."""
    flat = (np.asarray(mask).ravel(order="F") > 0).astype(np.uint8)
    padded = np.concatenate([[0], flat, [0]])
    changes = np.flatnonzero(padded[1:] != padded[:-1]) + 1
    changes[1::2] -= changes[0::2]
    return " ".join(map(str, changes))


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def read_annotations(csv_path):
    """Aceita ImageId/ClassId e o formato legado ImageId_ClassId."""
    annotations = {}
    with Path(csv_path).open(newline="", encoding="utf-8-sig") as stream:
        for row in csv.DictReader(stream):
            if "ImageId_ClassId" in row:
                image_id, class_id = row["ImageId_ClassId"].rsplit("_", 1)
            else:
                image_id, class_id = row["ImageId"], row["ClassId"]
            class_id = int(class_id)
            if class_id not in CLASSES:
                raise ValueError(f"Classe desconhecida: {class_id}")
            key = (image_id, class_id)
            if key in annotations:
                raise ValueError(f"Anotação duplicada: {key}")
            annotations[key] = (row.get("EncodedPixels") or "").strip()
    return annotations


def prepare_manifest(data_dir, output_dir, seed=42):
    """Divide imagens por grupos de hash, incluindo imagens sem defeitos."""
    data_dir, output_dir = Path(data_dir), Path(output_dir)
    image_dir = data_dir / "train_images"
    annotations = read_annotations(data_dir / "train.csv")
    paths = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {".jpg", ".png", ".jpeg"})
    if len(paths) < 3:
        raise ValueError("É necessário ter pelo menos três imagens")
    missing = {name for name, _ in annotations} - {p.name for p in paths}
    if missing:
        raise ValueError(f"Anotações sem imagem: {sorted(missing)[:5]}")
    records, groups = [], {}
    for path in paths:
        with Image.open(path) as image:
            width, height = image.size
            image.verify()
        rles = [annotations.get((path.name, c), "") for c in CLASSES]
        # Validar os limites antes de iniciar um treinamento demorado.
        for rle in rles:
            decode_rle(rle, height, width)
        record = {
            "image_id": path.name,
            "sha256": sha256_file(path),
            "height": height,
            "width": width,
            "rles": rles,
        }
        records.append(record)
        groups.setdefault(record["sha256"], []).append(record)
    hashes = sorted(groups)
    if len(hashes) < 3:
        raise ValueError("É necessário ter três grupos de imagens distintas")
    # Conteúdo idêntico com anotações incompatíveis deve ser revisado.
    for group in groups.values():
        if len({tuple(r["rles"]) for r in group}) > 1:
            raise ValueError("Imagens idênticas possuem anotações conflitantes")
    random.Random(seed).shuffle(hashes)
    n_test = max(1, round(len(hashes) * 0.15))
    n_val = max(1, round(len(hashes) * 0.15))
    assignments = {}
    for i, digest in enumerate(hashes):
        assignments[digest] = "test" if i < n_test else "val" if i < n_test + n_val else "train"
    for record in records:
        record["split"] = assignments[record["sha256"]]
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest = {
        "seed": seed,
        "split_method": "seeded_exact_file_hash_groups_70_15_15",
        "csv_sha256": sha256_file(data_dir / "train.csv"),
        "records": records,
    }
    target = output_dir / "manifest.json"
    target.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return target


class SteelDataset(Dataset):
    """Máscaras independentes por classe; aumentos sincronizados só no treino."""

    def __init__(self, data_dir, manifest_path, split, size=(128, 800), augment=False):
        self.image_dir = Path(data_dir) / "train_images"
        manifest = json.loads(Path(manifest_path).read_text())
        self.records = [r for r in manifest["records"] if r["split"] == split]
        self.size = tuple(size)
        self.augment = augment
        if not self.records:
            raise ValueError(f"Divisão vazia: {split}")
        if augment and split != "train":
            raise ValueError("Aumentos permitidos somente no treino")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        with Image.open(self.image_dir / record["image_id"]) as original:
            image = original.convert("RGB")
        if image.size != (record["width"], record["height"]):
            raise ValueError("Dimensões alteradas após a criação do manifesto")
        masks = [
            Image.fromarray(decode_rle(rle, record["height"], record["width"]))
            for rle in record["rles"]
        ]
        if self.augment:
            if random.random() < 0.5:
                image = ImageOps.mirror(image)
                masks = [ImageOps.mirror(m) for m in masks]
            if random.random() < 0.5:
                image = ImageOps.flip(image)
                masks = [ImageOps.flip(m) for m in masks]
            image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
        target_size = (self.size[1], self.size[0])
        image = image.resize(target_size, Image.Resampling.BILINEAR)
        x = np.asarray(image, dtype=np.float32).transpose(2, 0, 1) / 255.0
        y = np.stack([np.asarray(m.resize(target_size, Image.Resampling.NEAREST)) for m in masks])
        return (
            torch.from_numpy(x.copy()),
            torch.from_numpy(y.astype(np.float32)),
            record["image_id"],
        )
'''
(PACKAGE / 'data.py').write_text(source, encoding='utf-8')

### engine.py

In [ ]:
source = r'''"""Treinamento, seleção por validação e avaliação única no teste reservado."""

import csv
import json
import platform
import random
import subprocess
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from .data import SteelDataset, sha256_file
from .metrics import SegmentationMetrics
from .model import UNet, segmentation_loss
from .visualize import plot_history, save_comparison


@dataclass
class Config:
    epochs: int = 10
    batch_size: int = 4
    height: int = 128
    width: int = 800
    base_channels: int = 16
    learning_rate: float = 1e-3
    seed: int = 42
    num_workers: int = 0
    patience: int = 4
    threshold: float = 0.5
    device: str = "auto"
    data_kind: str = "severstal"
    mixed_precision: bool = True


def choose_device(value="auto"):
    if value != "auto":
        return torch.device(value)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)


def loader(data_dir, manifest, split, config, shuffle=False):
    dataset = SteelDataset(
        data_dir, manifest, split, (config.height, config.width), augment=shuffle
    )
    generator = torch.Generator().manual_seed(config.seed)
    return DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=shuffle,
        num_workers=config.num_workers,
        worker_init_fn=seed_worker,
        generator=generator,
        pin_memory=choose_device(config.device).type == "cuda",
        persistent_workers=config.num_workers > 0,
    )


@torch.inference_mode()
def evaluate_loader(model, batches, device, threshold):
    model.eval()
    meter = SegmentationMetrics()
    loss_sum = count = 0
    for image, target, names in batches:
        image, target = image.to(device), target.to(device)
        logits = model(image)
        loss_sum += float(segmentation_loss(logits, target)) * len(names)
        count += len(names)
        meter.update(logits.sigmoid().cpu().numpy(), target.cpu().numpy(), names, threshold)
    summary = meter.summary()
    summary["loss"] = loss_sum / count
    return summary, meter.rows


def git_revision():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL, text=True
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return None


def train(data_dir, manifest, output_dir, config):
    if config.epochs < 1 or config.patience < 1:
        raise ValueError("epochs e patience devem ser positivos")
    if config.height < 8 or config.width < 8 or not 0 < config.threshold < 1:
        raise ValueError("Dimensões >= 8 e limiar entre 0 e 1 são obrigatórios")
    output_dir = Path(output_dir)
    if (output_dir / "best.pt").exists():
        raise FileExistsError("Use outro diretório para preservar a execução anterior")
    output_dir.mkdir(parents=True, exist_ok=True)
    seed_everything(config.seed)
    device = choose_device(config.device)
    model = UNet(config.base_channels).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    use_amp = config.mixed_precision and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    train_batches = loader(data_dir, manifest, "train", config, shuffle=True)
    val_batches = loader(data_dir, manifest, "val", config)
    metadata = {
        "config": asdict(config),
        "device": str(device),
        "torch": str(torch.__version__),
        "python": platform.python_version(),
        "numpy": np.__version__,
        "git_commit": git_revision(),
        "manifest_sha256": sha256_file(manifest),
        "amp_enabled": use_amp,
        "note": "Synthetic smoke test; not a Severstal result"
        if config.data_kind == "synthetic"
        else "Baseline; no production deployment claim",
    }
    (output_dir / "run.json").write_text(json.dumps(metadata, indent=2))
    # Copiar o manifesto fixa o split e as anotações usados nesta execução.
    (output_dir / "manifest.json").write_text(Path(manifest).read_text())
    best_score, best_loss, stale, history = -1.0, float("inf"), 0, []
    started = time.perf_counter()
    for epoch in range(1, config.epochs + 1):
        model.train()
        loss_sum = count = 0
        epoch_started = time.perf_counter()
        for step, (image, target, _) in enumerate(train_batches, start=1):
            image = image.to(device, non_blocking=True)
            target = target.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                loss = segmentation_loss(model(image), target)
            if not torch.isfinite(loss):
                raise RuntimeError("Loss não finita; verificar entradas e taxa de aprendizado")
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            loss_sum += float(loss.detach()) * image.shape[0]
            count += image.shape[0]
            if step % 100 == 0 or step == len(train_batches):
                print(
                    json.dumps(
                        {
                            "phase": "train",
                            "epoch": epoch,
                            "step": step,
                            "steps": len(train_batches),
                            "images_seen": count,
                            "elapsed_seconds": round(time.perf_counter() - epoch_started, 1),
                        }
                    ),
                    flush=True,
                )
        val, _ = evaluate_loader(model, val_batches, device, config.threshold)
        score = val["macro_dice"] if val["macro_dice"] is not None else 0.0
        row = {
            "epoch": epoch,
            "train_loss": loss_sum / count,
            "val_loss": val["loss"],
            "val_macro_dice": score,
        }
        history.append(row)
        print(json.dumps(row), flush=True)
        # Val loss desempata Dice=0 no início sem consultar o teste.
        if score > best_score or (score == best_score and val["loss"] < best_loss):
            best_score, best_loss, stale = score, val["loss"], 0
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "config": asdict(config),
                    "epoch": epoch,
                    "val_metrics": val,
                    "manifest_sha256": metadata["manifest_sha256"],
                },
                output_dir / "best.pt",
            )
        else:
            stale += 1
        (output_dir / "history.json").write_text(json.dumps(history, indent=2))
        if stale >= config.patience:
            break
    metadata["elapsed_seconds"] = time.perf_counter() - started
    metadata["best_val_macro_dice"] = best_score
    (output_dir / "run.json").write_text(json.dumps(metadata, indent=2))
    plot_history(history, output_dir / "learning_curves.png")
    return output_dir / "best.pt"


def load_checkpoint(checkpoint_path, device="auto"):
    device = choose_device(device)
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    config = Config(**checkpoint["config"])
    model = UNet(config.base_channels)
    model.load_state_dict(checkpoint["model_state"])
    return model.to(device).eval(), config, checkpoint, device


@torch.inference_mode()
def evaluate(data_dir, manifest, checkpoint_path, output_dir, split="test", device="auto"):
    output_dir = Path(output_dir)
    if (output_dir / "metrics.json").exists():
        raise FileExistsError("Avaliação já existe; preserve o teste reservado")
    model, config, checkpoint, selected_device = load_checkpoint(checkpoint_path, device)
    if sha256_file(manifest) != checkpoint["manifest_sha256"]:
        raise ValueError("O manifesto difere daquele usado no treinamento")
    batches = loader(data_dir, manifest, split, config)
    summary, rows = evaluate_loader(model, batches, selected_device, config.threshold)
    summary.update(
        {
            "split": split,
            "threshold": config.threshold,
            "data_kind": config.data_kind,
            "checkpoint_sha256": sha256_file(checkpoint_path),
            "evaluation_resolution": [config.height, config.width],
            "best_epoch": checkpoint["epoch"],
        }
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "metrics.json").write_text(json.dumps(summary, indent=2, allow_nan=False))
    with (output_dir / "per_image_class.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)
    # Apenas erros efetivos, evitando mostrar vazios corretos como falhas.
    failures = sorted(
        [r for r in rows if r["fp_pixels"] + r["fn_pixels"] > 0],
        key=lambda r: (r["dice"], -r["fn_pixels"] - r["fp_pixels"]),
    )
    (output_dir / "worst_errors.json").write_text(json.dumps(failures[:30], indent=2))
    dataset = batches.dataset
    by_name = {r["image_id"]: i for i, r in enumerate(dataset.records)}
    chosen = list(dict.fromkeys(r["image_id"] for r in failures))[:6]
    if not chosen:
        chosen = [r["image_id"] for r in dataset.records[:3]]
    for n, name in enumerate(chosen):
        image, target, _ = dataset[by_name[name]]
        probabilities = model(image.unsqueeze(0).to(selected_device)).sigmoid()[0].cpu().numpy()
        save_comparison(
            image.numpy().transpose(1, 2, 0),
            target.numpy(),
            probabilities,
            output_dir / f"prediction_{n:02d}.png",
            f"{config.data_kind} | {split} | {name}",
            config.threshold,
        )
    lines = [
        "# Relatório de avaliação",
        "",
        f"Dados: **{config.data_kind}**. Divisão: **{split}**.",
        "",
        "Resultados em resolução de avaliação reduzida; não equivalem ao placar Kaggle.",
        "",
        "| Classe | Dice global | IoU | Recall por pixel | Dice imagens positivas |",
        "|---|---:|---:|---:|---:|",
    ]

    def fmt(x):
        return "n/a" if x is None else f"{x:.4f}"

    for item in summary["classes"]:
        lines.append(
            f"| {item['class_id']} | {fmt(item['dice'])} | {fmt(item['iou'])} | "
            f"{fmt(item['pixel_recall'])} | {fmt(item['positive_image_mean_dice'])} |"
        )
    lines += [
        "",
        "O checkpoint foi selecionado apenas na validação. Limiar fixo em 0,5.",
        "Máscaras vazias corretas não elevam o Dice global por classe.",
        "Consulte per_image_class.csv, worst_errors.json e prediction_*.png.",
    ]
    if config.data_kind == "synthetic":
        lines += [
            "",
            "**DEMONSTRAÇÃO SINTÉTICA: estas métricas não medem desempenho em aço real.**",
        ]
    (output_dir / "report.md").write_text("\n".join(lines), encoding="utf-8")
    return summary


@torch.inference_mode()
def predict(image_path, checkpoint_path, output_dir, device="auto"):
    from PIL import Image

    from .data import encode_rle

    model, config, _, device = load_checkpoint(checkpoint_path, device)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    with Image.open(image_path) as image:
        original_size = image.size
        resized = image.convert("RGB").resize(
            (config.width, config.height), Image.Resampling.BILINEAR
        )
    array = np.asarray(resized, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(array.transpose(2, 0, 1).copy()).unsqueeze(0).to(device)
    logits = model(tensor)
    # Restaurar probabilidades à resolução original antes de aplicar limiar.
    probabilities = torch.nn.functional.interpolate(
        logits.sigmoid(),
        size=(original_size[1], original_size[0]),
        mode="bilinear",
        align_corners=False,
    )
    masks = probabilities[0].cpu().numpy() >= config.threshold
    np.savez_compressed(output_dir / "prediction.npz", masks=masks)
    payload = {
        "image_id": Path(image_path).name,
        "width": original_size[0],
        "height": original_size[1],
        "threshold": config.threshold,
        "checkpoint_sha256": sha256_file(checkpoint_path),
        "classes": [{"class_id": k + 1, "encoded_pixels": encode_rle(masks[k])} for k in range(4)],
    }
    (output_dir / "prediction.json").write_text(json.dumps(payload, indent=2))
    display_prob = logits.sigmoid()[0].cpu().numpy()
    save_comparison(
        array,
        None,
        display_prob,
        output_dir / "prediction.png",
        "Inferência: sem anotação de referência",
        config.threshold,
    )
    return payload
'''
(PACKAGE / 'engine.py').write_text(source, encoding='utf-8')

### metrics.py

In [ ]:
source = r'''"""Métricas globais por classe e Dice separado em imagens positivas."""

import numpy as np


def ratio(numerator, denominator):
    return float(numerator / denominator) if denominator else None


class SegmentationMetrics:
    def __init__(self):
        self.tp = np.zeros(4, dtype=np.int64)
        self.fp = self.tp.copy()
        self.fn = self.tp.copy()
        self.positive_dice = [[] for _ in range(4)]
        self.negative_images = self.tp.copy()
        self.false_positive_images = self.tp.copy()
        self.positive_images = self.tp.copy()
        self.detected_positive_images = self.tp.copy()
        self.rows = []

    def update(self, probabilities, target, image_ids, threshold=0.5):
        thresholds = np.asarray(threshold)
        if thresholds.ndim == 1:
            if thresholds.shape != (4,):
                raise ValueError("Forneça quatro limiares ou um escalar")
            thresholds = thresholds.reshape(1, 4, 1, 1)
        pred = np.asarray(probabilities) >= thresholds
        true = np.asarray(target) > 0
        if pred.shape != true.shape or pred.ndim != 4 or pred.shape[1] != 4:
            raise ValueError("Esperado [batch, 4, altura, largura] para predição e alvo")
        for i, image_id in enumerate(image_ids):
            for k in range(4):
                p, t = pred[i, k], true[i, k]
                tp, fp, fn = int((p & t).sum()), int((p & ~t).sum()), int((~p & t).sum())
                self.tp[k] += tp
                self.fp[k] += fp
                self.fn[k] += fn
                denom = 2 * tp + fp + fn
                dice = 2 * tp / denom if denom else 1.0
                if t.any():
                    self.positive_dice[k].append(dice)
                    self.positive_images[k] += 1
                    self.detected_positive_images[k] += int(p.any())
                else:
                    self.negative_images[k] += 1
                    self.false_positive_images[k] += int(p.any())
                self.rows.append(
                    {
                        "image_id": image_id,
                        "class_id": k + 1,
                        "dice": dice,
                        "tp_pixels": tp,
                        "fp_pixels": fp,
                        "fn_pixels": fn,
                        "true_positive_pixels": int(t.sum()),
                    }
                )

    def summary(self):
        classes = []
        for k in range(4):
            tp, fp, fn = int(self.tp[k]), int(self.fp[k]), int(self.fn[k])
            classes.append(
                {
                    "class_id": k + 1,
                    "dice": ratio(2 * tp, 2 * tp + fp + fn),
                    "iou": ratio(tp, tp + fp + fn),
                    "pixel_precision": ratio(tp, tp + fp),
                    "pixel_recall": ratio(tp, tp + fn),
                    "positive_image_mean_dice": float(np.mean(self.positive_dice[k]))
                    if self.positive_dice[k]
                    else None,
                    "positive_images": int(self.positive_images[k]),
                    "image_presence_recall": ratio(
                        self.detected_positive_images[k], self.positive_images[k]
                    ),
                    "negative_images": int(self.negative_images[k]),
                    "negative_image_fp_rate": ratio(
                        self.false_positive_images[k], self.negative_images[k]
                    ),
                    "tp_pixels": tp,
                    "fp_pixels": fp,
                    "fn_pixels": fn,
                }
            )
        valid = [c["dice"] for c in classes if c["dice"] is not None]
        return {
            "classes": classes,
            "macro_dice": float(np.mean(valid)) if valid else None,
            "valid_classes": len(valid),
            "conventions": "global pixel counts; absent global class -> null; "
            "per-image empty/empty Dice=1; positive-image Dice excludes empty targets; "
            "image presence means at least one predicted positive pixel",
        }
'''
(PACKAGE / 'metrics.py').write_text(source, encoding='utf-8')

### model.py

In [ ]:
source = r'''"""U-Net compacta treinada do zero; quatro canais sigmoid independentes."""

import torch
from torch import nn
from torch.nn import functional as F


def block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, 3, padding=1),
        nn.GroupNorm(4, out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, 3, padding=1),
        nn.GroupNorm(4, out_channels),
        nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    def __init__(self, base_channels=16):
        super().__init__()
        b = base_channels
        if b < 4 or b % 4:
            raise ValueError("base_channels deve ser múltiplo de 4")
        self.encoders = nn.ModuleList([block(3, b), block(b, b * 2), block(b * 2, b * 4)])
        self.bottleneck = block(b * 4, b * 8)
        self.decoders = nn.ModuleList([block(b * 12, b * 4), block(b * 6, b * 2), block(b * 3, b)])
        self.head = nn.Conv2d(b, 4, 1)

    def forward(self, x):
        skips = []
        for encoder in self.encoders:
            x = encoder(x)
            skips.append(x)
            x = F.max_pool2d(x, 2)
        x = self.bottleneck(x)
        for decoder, skip in zip(self.decoders, reversed(skips)):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = decoder(torch.cat([x, skip], dim=1))
        return self.head(x)


def segmentation_loss(logits, target, pos_weight=None):
    """BCE + soft Dice; sigmoid não deve ser aplicada antes da BCEWithLogits."""
    # A soma espacial pode ultrapassar 65504 em FP16; a loss permanece em FP32.
    bce = F.binary_cross_entropy_with_logits(logits.float(), target.float(), pos_weight=pos_weight)
    prob = logits.float().sigmoid()
    dims = (0, 2, 3)
    dice = (2 * (prob * target).sum(dims) + 1) / (prob.sum(dims) + target.sum(dims) + 1)
    return bce + 1 - dice.mean()
'''
(PACKAGE / 'model.py').write_text(source, encoding='utf-8')

### synthetic.py

In [ ]:
source = r'''"""Gera imagens geométricas para testar execução; não simula aço real."""

import csv
from pathlib import Path

import numpy as np
from PIL import Image

from .data import encode_rle


def generate_synthetic(output_dir, count=40, seed=42):
    output_dir = Path(output_dir)
    if (output_dir / "train.csv").exists():
        raise FileExistsError("Dataset já existe; escolha outro diretório")
    if count < 12:
        raise ValueError("Use pelo menos 12 exemplos sintéticos")
    image_dir = output_dir / "train_images"
    image_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    height, width = 64, 256
    rows = []
    colors = [(220, 60, 60), (60, 220, 60), (60, 60, 220), (230, 200, 50)]
    for i in range(count):
        image = rng.integers(35, 70, (height, width, 3), dtype=np.uint8)
        masks = np.zeros((4, height, width), dtype=np.uint8)
        # 20% de imagens normais; algumas imagens multiclasse.
        selected = [] if i % 5 == 0 else [i % 4]
        if i % 7 == 0 and selected:
            selected.append((i + 1) % 4)
        for k in selected:
            top, left = int(rng.integers(5, 35)), int(rng.integers(5, 210))
            masks[k, top : top + 15, left : left + 30] = 1
            image[masks[k] > 0] = colors[k]
        name = f"synthetic_{i:04d}.png"
        Image.fromarray(image).save(image_dir / name)
        for k in range(4):
            rows.append({"ImageId": name, "ClassId": k + 1, "EncodedPixels": encode_rle(masks[k])})
    with (output_dir / "train.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=["ImageId", "ClassId", "EncodedPixels"])
        writer.writeheader()
        writer.writerows(rows)
    (output_dir / "SYNTHETIC.txt").write_text(
        "Dados geométricos artificiais para smoke test. Não são imagens Severstal ou aço real.\n"
    )
    return output_dir
'''
(PACKAGE / 'synthetic.py').write_text(source, encoding='utf-8')

### transfer.py

In [ ]:
source = r'''"""Encoder ResNet18 opcional com normalização ImageNet incorporada ao modelo."""

import torch
from torch import nn
from torch.nn import functional as F

from .model import UNet, block


class ResNetUNet(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        from torchvision.models import ResNet18_Weights, resnet18

        encoder = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool = encoder.maxpool
        self.layers = nn.ModuleList(
            [encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4]
        )
        self.decoders = nn.ModuleList(
            [block(768, 128), block(256, 64), block(128, 32), block(96, 32)]
        )
        self.head = nn.Conv2d(32, 4, 1)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1))

    def train(self, mode=True):
        super().train(mode)
        # Estatísticas do encoder ficam estáveis em lotes pequenos; decoder usa GroupNorm.
        for module in [self.stem, *self.layers]:
            for layer in module.modules():
                if isinstance(layer, nn.BatchNorm2d):
                    layer.eval()
        return self

    def forward(self, image):
        shape = image.shape[-2:]
        first = self.stem((image - self.mean) / self.std)
        features, x = [first], self.pool(first)
        for layer in self.layers:
            x = layer(x)
            features.append(x)
        x = features[-1]
        for decoder, skip in zip(self.decoders, reversed(features[:-1])):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            x = decoder(torch.cat([x, skip], 1))
        return self.head(F.interpolate(x, size=shape, mode="bilinear", align_corners=False))


def build_model(config, initialize_pretrained=False):
    if config.architecture == "resnet18_unet":
        return ResNetUNet(pretrained=initialize_pretrained and config.pretrained)
    if config.architecture == "unet":
        return UNet(config.base_channels)
    raise ValueError(f"Arquitetura desconhecida: {config.architecture}")
'''
(PACKAGE / 'transfer.py').write_text(source, encoding='utf-8')

### visualize.py

In [ ]:
source = r'''"""Análise exploratória, curvas e comparação visual de máscaras."""

import json
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

COLORS = np.array([[1.0, 0.75, 0.15], [0.7, 0.35, 1.0], [0.1, 0.85, 0.8], [1.0, 0.25, 0.4]])


def overlay(image, masks):
    result = image.copy()
    for channel, color in zip(masks, COLORS):
        selected = channel > 0
        result[selected] = 0.55 * result[selected] + 0.45 * color
    return np.clip(result, 0, 1)


def save_comparison(image, target, probabilities, output, title, threshold=0.5):
    arrays, labels = [image], ["Imagem"]
    if target is not None:
        arrays.append(overlay(image, target))
        labels.append("Máscara real")
    if probabilities is not None:
        arrays.append(overlay(image, probabilities >= threshold))
        labels.append("Predição")
    fig, axes = plt.subplots(1, len(arrays), figsize=(5 * len(arrays), 4), constrained_layout=True)
    for ax, array, label in zip(np.atleast_1d(axes), arrays, labels):
        ax.imshow(array)
        ax.set_title(label)
        ax.axis("off")
    fig.suptitle(title + " | classes: 1 dourado · 2 roxo · 3 ciano · 4 rosa", fontsize=10)
    fig.savefig(output, dpi=140)
    plt.close(fig)


def explore(data_dir, manifest_path, output_dir, size=(128, 800)):
    from .data import SteelDataset

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest = json.loads(Path(manifest_path).read_text())
    records = manifest["records"]
    summary = {
        "images": len(records),
        "unique_file_hashes": len({r["sha256"] for r in records}),
        "splits": {},
        "resolution_counts": {},
    }
    for record in records:
        key = f"{record['width']}x{record['height']}"
        summary["resolution_counts"][key] = summary["resolution_counts"].get(key, 0) + 1
    for split in ["train", "val", "test"]:
        selected = [r for r in records if r["split"] == split]
        summary["splits"][split] = {
            "images": len(selected),
            "no_defect": sum(not any(r["rles"]) for r in selected),
            "positive_images_per_class": {
                str(k + 1): sum(bool(r["rles"][k]) for r in selected) for k in range(4)
            },
        }
    (output_dir / "data_summary.json").write_text(json.dumps(summary, indent=2))
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    for j, split in enumerate(["train", "val", "test"]):
        counts = summary["splits"][split]["positive_images_per_class"]
        ax.bar(np.arange(4) + j * 0.25, [counts[str(k)] for k in range(1, 5)], 0.25, label=split)
    ax.set_xticks(np.arange(4) + 0.25, ["Classe 1", "Classe 2", "Classe 3", "Classe 4"])
    ax.set_ylabel("Imagens positivas (classes podem coexistir)")
    ax.legend()
    fig.savefig(output_dir / "class_distribution.png", dpi=140)
    plt.close(fig)
    dataset = SteelDataset(data_dir, manifest_path, "train", size=size)
    # EDA visual usa apenas treino; os exemplos incluem positivos e negativos quando disponíveis.
    candidates = sorted(
        range(len(dataset)), key=lambda i: -sum(bool(v) for v in dataset.records[i]["rles"])
    )
    chosen = candidates[:3]
    negatives = [i for i in candidates if not any(dataset.records[i]["rles"])]
    if negatives and negatives[0] not in chosen:
        chosen.append(negatives[0])
    for i in chosen:
        image, mask, name = dataset[i]
        save_comparison(
            image.numpy().transpose(1, 2, 0),
            mask.numpy(),
            None,
            output_dir / f"example_{i}.png",
            f"EDA treino: {name}",
        )
    return summary


def plot_history(history, output):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    axes[0].plot([r["epoch"] for r in history], [r["train_loss"] for r in history], label="treino")
    if any(r["val_loss"] is not None for r in history):
        axes[0].plot(
            [r["epoch"] for r in history], [r["val_loss"] for r in history], label="validação"
        )
    axes[0].set_title("BCE + soft Dice")
    axes[0].legend()
    axes[1].plot([r["epoch"] for r in history], [r["val_macro_dice"] for r in history])
    axes[1].set_title("Macro Dice de validação · limiar fixo 0,5")
    for ax in axes:
        ax.set_xlabel("Época")
    fig.savefig(output, dpi=140)
    plt.close(fig)
'''
(PACKAGE / 'visualize.py').write_text(source, encoding='utf-8')

## 3. Integridade e dados
Se o manifesto já existir, ele é reutilizado e verificado: não recalcular hashes após uma alteração
para ocultar a divergência. Nenhum recorte atravessa os splits de imagens de origem.


In [ ]:
import sys, json
import torch
from PIL import Image as PILImage
sys.path.insert(0, str(PACKAGE_ROOT))
from steelvision.data import prepare_manifest
from steelvision.advanced import AdvancedConfig, train_advanced, calibrate, evaluate_advanced, predict_advanced
from steelvision.advanced_data import verify_integrity, training_statistics, records_for
from steelvision.synthetic import generate_synthetic

if USE_SYNTHETIC:
    DATA_DIR = OUTPUT / "synthetic_data"
    if not DATA_DIR.exists():
        generate_synthetic(DATA_DIR, count=16)
    torch.set_num_threads(2)
else:
    candidates = sorted(p.parent for p in Path("/kaggle/input").rglob("train.csv")
                        if (p.parent / "train_images").is_dir())
    if DATA_DIR_OVERRIDE is None and len(candidates) != 1:
        raise RuntimeError("Anexe os dados da competição ou configure DATA_DIR_OVERRIDE")
    DATA_DIR = Path(DATA_DIR_OVERRIDE) if DATA_DIR_OVERRIDE is not None else candidates[0]
manifest = OUTPUT / "prepared" / "manifest.json"
if not manifest.exists():
    manifest = prepare_manifest(DATA_DIR, manifest.parent, seed=42)
print(verify_integrity(DATA_DIR, manifest, {"train", "val"}))
statistics = training_statistics(manifest)
print(json.dumps({k:v for k,v in statistics.items() if k != "sample_weights"}, indent=2))


## 4. Treino com pesos e amostragem exclusivos do treino
Patches de 256 × 400 preservam a escala original. Metade dos recortes de imagens positivas tenta incluir
um pixel anotado; os demais são uniformes. Imagens normais continuam presentes.
O encoder pré-treinado recebe normalização ImageNet incorporada; suas estatísticas BatchNorm ficam
congeladas para lotes pequenos. A loss e as reduções espaciais permanecem em FP32 sob AMP.

`last.pt` salva modelo, otimizador, GradScaler, RNGs, histórico e critério de parada a cada época.
A retomada verifica manifesto, código e configuração. `best.pt` preserva o checkpoint de melhor
macro Dice de validação nativa com limiar fixo 0,5.


In [ ]:
config = AdvancedConfig(
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    patch_height=32 if USE_SYNTHETIC else 256,
    patch_width=64 if USE_SYNTHETIC else 400,
    architecture="unet" if USE_SYNTHETIC else ARCHITECTURE,
    pretrained=False if USE_SYNTHETIC else PRETRAINED,
    base_channels=4 if USE_SYNTHETIC else 16,
    num_workers=0 if USE_SYNTHETIC else 2,
    device="cpu" if USE_SYNTHETIC else "auto",
    data_kind="synthetic" if USE_SYNTHETIC else "severstal",
    balanced_sampling=BALANCED_SAMPLING, weighted_loss=WEIGHTED_LOSS)
checkpoint = train_advanced(DATA_DIR, manifest, OUTPUT / "run", config, resume=RESUME_FROM)
print("Melhor checkpoint:", checkpoint)
with PILImage.open(OUTPUT / "run" / "learning_curves.png") as image:
    display(image.copy())


## 5. Calibração somente na validação
Buscar por classe o limiar de maior Dice global em grade 0,05…0,95. Empates preferem o limiar maior.
A busca acumula contagens sem guardar todas as probabilidades. Classes sem suporte positivo mantêm 0,5.
O modelo não é retreinado nesta etapa. Os limiares e o hash da calibração acompanham `calibrated.pt`.


In [ ]:
deployment_checkpoint = calibrate(DATA_DIR, manifest, checkpoint, OUTPUT / "calibration")
print((OUTPUT / "calibration" / "calibration.json").read_text())


## 6. Avaliação uma vez no teste histórico
O caminho de inferência é o mesmo usado na validação: mosaico nativo, média nas sobreposições e limiares
congelados. Não comparar diretamente este Dice nativo com o Dice em resolução reduzida do baseline
como se a única diferença fosse o modelo. Uma comparação controlada requer reavaliar o baseline no
mesmo caminho nativo, sem recalibrar seus limiares no teste.

Este teste histórico já foi visto em outro experimento. Não chamá-lo de validação externa ou nova
estimativa independente. As próximas escolhas de modelo continuam limitadas à validação.


In [ ]:
metrics = evaluate_advanced(DATA_DIR, manifest, deployment_checkpoint, OUTPUT / "historical_test")
print(json.dumps(metrics, indent=2))
for path in sorted((OUTPUT / "historical_test").glob("prediction_*.png")):
    with PILImage.open(path) as image:
        display(image.copy())


## 7. Inferência isolada com os mesmos limiares e resolução

In [ ]:
example = records_for(manifest, "test")[0]["image_id"]
result = predict_advanced(DATA_DIR / "train_images" / example, deployment_checkpoint, OUTPUT / "inference")
print("Limiares publicados:", result["thresholds"])
with PILImage.open(OUTPUT / "inference" / "prediction.png") as image:
    display(image.copy())


## 8. Artefatos
O ZIP é para armazenamento no Kaggle/local. Não enviar imagens, máscaras ou manifesto RLE para o GitHub.
Publicar ali apenas métricas, configurações e gráficos agregados. ONNX, incidentes e API/MLflow são os
laboratórios 2–4. Os posts no LinkedIn serão preparados após os quatro laboratórios, com evidências reais.


In [ ]:
import zipfile
archive = BASE / f"steelvision_{RUN_NAME}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file() and "synthetic_data" not in path.relative_to(OUTPUT).parts:
            bundle.write(path, path.relative_to(OUTPUT))
print("Artefatos:", archive)
